In [2]:
import pandas as pd

df = pd.read_csv("../data/customer_support_tickets_clean.csv")

df.head()

,subject,body,answer,type,queue,priority,ticket_text,clean_text
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support team ...
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer support i hope thi...


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

X = df["clean_text"]
y = df["queue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 5, 10, 20]
}

In [5]:
grid_search = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_tfidf, y_train)

,estimator,LinearSVC(random_state=42)
,param_grid,"{'C': [0.1, 1, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [6]:
best_svm = grid_search.best_estimator_

best_svm

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,5
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [7]:
y_pred_best = best_svm.predict(X_test_tfidf)

In [8]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_best = accuracy_score(y_test, y_pred_best)

print("Optimized Accuracy:", accuracy_best)

print(classification_report(y_test, y_pred_best))

Optimized Accuracy: 0.7153351698806244
                                 precision    recall  f1-score   support

           Billing and Payments       0.84      0.89      0.86       270
               Customer Service       0.65      0.68      0.66       499
                General Inquiry       0.91      0.68      0.78        44
                Human Resources       0.86      0.57      0.69        77
                     IT Support       0.67      0.66      0.67       363
                Product Support       0.68      0.65      0.66       628
          Returns and Exchanges       0.82      0.60      0.70       162
            Sales and Pre-Sales       0.80      0.58      0.67       124
Service Outages and Maintenance       0.88      0.77      0.82       156
              Technical Support       0.70      0.79      0.74       944

                       accuracy                           0.72      3267
                      macro avg       0.78      0.69      0.73      3267
          

In [9]:
balanced_svm = LinearSVC(
    C=5,
    class_weight="balanced",
    random_state=42
)

balanced_svm.fit(X_train_tfidf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,5
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [10]:
y_pred_balanced = balanced_svm.predict(X_test_tfidf)

In [11]:
accuracy_balanced = accuracy_score(y_test, y_pred_balanced)

print("Balanced SVM Accuracy:", accuracy_balanced)

print(classification_report(y_test, y_pred_balanced))

Balanced SVM Accuracy: 0.7144168962350781
                                 precision    recall  f1-score   support

           Billing and Payments       0.84      0.89      0.86       270
               Customer Service       0.64      0.68      0.66       499
                General Inquiry       0.82      0.73      0.77        44
                Human Resources       0.84      0.60      0.70        77
                     IT Support       0.65      0.67      0.66       363
                Product Support       0.68      0.64      0.66       628
          Returns and Exchanges       0.78      0.67      0.72       162
            Sales and Pre-Sales       0.77      0.64      0.70       124
Service Outages and Maintenance       0.85      0.80      0.83       156
              Technical Support       0.72      0.77      0.74       944

                       accuracy                           0.71      3267
                      macro avg       0.76      0.71      0.73      3267
       

In [12]:
from sklearn.linear_model import SGDClassifier

sgd_model = SGDClassifier(
    loss="hinge",
    random_state=42
)

sgd_model.fit(X_train_tfidf, y_train)

,loss,'hinge'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,1000
,tol,0.001
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,None


In [13]:
y_pred_sgd = sgd_model.predict(X_test_tfidf)

In [14]:
accuracy_sgd = accuracy_score(y_test, y_pred_sgd)

print("SGD Accuracy:", accuracy_sgd)

print(classification_report(y_test, y_pred_sgd))

SGD Accuracy: 0.6293235384144475
                                 precision    recall  f1-score   support

           Billing and Payments       0.78      0.84      0.81       270
               Customer Service       0.60      0.53      0.56       499
                General Inquiry       0.78      0.41      0.54        44
                Human Resources       0.77      0.35      0.48        77
                     IT Support       0.69      0.48      0.56       363
                Product Support       0.58      0.51      0.54       628
          Returns and Exchanges       0.77      0.49      0.60       162
            Sales and Pre-Sales       0.74      0.44      0.56       124
Service Outages and Maintenance       0.81      0.75      0.78       156
              Technical Support       0.57      0.82      0.68       944

                       accuracy                           0.63      3267
                      macro avg       0.71      0.56      0.61      3267
                

In [15]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(best_svm, "../models/queue_model.pkl")
joblib.dump(vectorizer, "../models/queue_vectorizer.pkl")

print("✅ Queue model saved successfully!")

✅ Queue model saved successfully!
